# 03 · Bayesian Beta-Binomial — `foe.bayesian`

**Hexkit surfaces:** `pages/sequential_analysis.py` (revenue-impact overlay),
`pages/experiment_analysis.py`, `utility/automation_engine.py`. Hexkit has no Bayesian math of
its own: these pages build the inputs and call `foe.bayesian.operations` (`BayesianEngine`,
`get_lift_prior`), which is what this notebook validates.

FOE estimates everything by Monte Carlo: Beta(1 + x, 1 + n − x) posteriors, optionally
re-weighted by a normal **lift prior** on `log(p_B / p_A)`. With several challengers the lift
prior applies to each challenger's lift vs control, and the joint weight is the product of those
densities. `lift_prior=None` means no lift prior: the plain Beta-Binomial posterior. The R side
computes the same quantities **deterministically** by nested numerical integration
(`integrate`) over the posteriors, with the lift-prior densities as weights. Differences are
therefore pure Monte Carlo error, and the tolerances are set at a few Monte Carlo standard
errors.

| FOE function | R reference |
|---|---|
| `run_probability_analysis` — P(B > A), E[(B − A)⁺], E[(A − B)⁺], with no lift prior and with lift priors | nested `integrate` over `dbeta × dbeta`, weighted by `dnorm(log b − log a)` when there is a lift prior |
| `run_probability_analysis` — P(best) for 3 arms, no lift prior | `integrate(dbeta_i(x) · Π pbeta_j(x))` |
| `run_probability_analysis` — P(best) for 3 arms, skeptical lift prior | triple nested `integrate` over `dbeta(a)·dbeta(b)·dbeta(c)·dnorm(log b − log a)·dnorm(log c − log a)` |
| `run_monetary_projection` — P(B > A), P(revenue better), expected uplift / risk; equal and unequal traffic, equal and unequal AOVs, with and without a lift prior | exact integral: `period · E_w[((b·AOV_B − a·AOV_A) · mean(n_A, n_B) / runtime)⁺]` (AOV noise is a shared, mean-preserving factor) |
| `get_lift_prior` | `log1p` |

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd

from foe.bayesian.operations import BayesianEngine, get_lift_prior
from foe.core.models import BusinessCaseInput, ExperimentInput
from validation_helpers import Checks

checks = Checks()
N_SAMPLES = 1_000_000

## R helper: weighted expectations by nested integration

In [2]:
%%R
# E_w[h(a, b)] over the region lo(a) < b < hi(a), where a ~ Beta(pa), b ~ Beta(pb) and the
# weight is the lift-prior density dnorm(log b - log a; m, s). s = NA means no lift prior
# (weight 1). Unnormalised; divide by the same integral with h = 1 over the full range.
nested <- function(pa, pb, m, s, h = function(a, b) 1,
                   lo = function(a) -Inf, hi = function(a) Inf) {
  w <- function(a, b) if (is.na(s)) 1 else dnorm(log(b) - log(a), m, s)
  ra <- qbeta(c(1e-10, 1 - 1e-10), pa[1], pa[2])
  rb <- qbeta(c(1e-10, 1 - 1e-10), pb[1], pb[2])
  inner <- function(a) {
    l <- max(lo(a), rb[1]); u <- min(hi(a), rb[2])
    if (l >= u) return(0)
    integrate(function(b) h(a, b) * dbeta(b, pb[1], pb[2]) * w(a, b),
              l, u, rel.tol = 1e-10)$value
  }
  integrate(function(a) dbeta(a, pa[1], pa[2]) * sapply(a, inner),
            ra[1], ra[2], rel.tol = 1e-8)$value
}

# P(B > A), E[(B-A)+], E[(A-B)+] under the lift prior, for one control/challenger pair.
pair_summary <- function(xa, na, xb, nb, m, s) {
  pa <- c(1 + xa, 1 + na - xa); pb <- c(1 + xb, 1 + nb - xb)
  z <- nested(pa, pb, m, s)
  data.frame(
    prob_beat_control = nested(pa, pb, m, s, lo = function(a) a) / z,
    expected_uplift   = nested(pa, pb, m, s, h = function(a, b) b - a, lo = function(a) a) / z,
    expected_loss     = nested(pa, pb, m, s, h = function(a, b) a - b, hi = function(a) a) / z
  )
}

## 1. Two-arm probabilities: no lift prior, uninformative, skeptical and moderate lift priors

In [3]:
cases = [
    # name, control n, control x, challenger n, challenger x, expected lift %, skepticism
    # (skepticism None = lift_prior=None, i.e. no lift prior)
    ("small positive, no lift prior", 10_000, 300, 10_000, 335,  0.0, None),
    ("small positive, uninformative", 10_000, 300, 10_000, 335,  0.0, "uninformative"),
    ("small positive, skeptical",     10_000, 300, 10_000, 335,  0.0, "skeptical"),
    ("large lift, moderate +5%",       4_000, 120,  4_000, 170,  5.0, "moderate"),
    ("negative, skeptical",           20_000, 900, 20_000, 840,  0.0, "skeptical"),
]
rows = []
for name, na, xa, nb, xb, lift, skep in cases:
    prior = None if skep is None else get_lift_prior(lift, skep)
    r = BayesianEngine(seed=1).run_probability_analysis(
        ExperimentInput(visitors=[na, nb], conversions=[xa, xb]),
        lift_prior=prior, n_samples=N_SAMPLES,
    )[0]
    rows.append(dict(name=name, na=na, xa=xa, nb=nb, xb=xb,
                     m=prior.mean_log_lift if prior else np.nan,
                     s=prior.std_log_lift if prior else np.nan, lift=lift,
                     prob_beat_control=r.prob_beat_control,
                     expected_uplift=r.expected_uplift, expected_loss=r.expected_loss))
py = pd.DataFrame(rows)
py

,name,na,xa,nb,xb,m,s,lift,prob_beat_control,expected_uplift,expected_loss
0,"small positive, no lift prior",10000,300,10000,335,NaN,NaN,0.0,0.920780,0.003587,0.000089
1,"small positive, uninformative",10000,300,10000,335,0.00000,1.00,0.0,0.920146,0.003567,0.000090
2,"small positive, skeptical",10000,300,10000,335,0.00000,0.10,0.0,0.866922,0.002305,0.000132
3,"large lift, moderate +5%",4000,120,4000,170,0.04879,0.25,5.0,0.997559,0.010612,0.000003
4,"negative, skeptical",20000,900,20000,840,0.00000,0.10,0.0,0.091657,0.000079,0.002539


In [4]:
%%R -i py -o r_out
r_out <- do.call(rbind, lapply(seq_len(nrow(py)), function(i) {
  s <- py[i, ]
  pair_summary(s$xa, s$na, s$xb, s$nb, s$m, s$s)
}))
r_out$m_check <- log1p(py$lift / 100)
r_out

 prob_beat_control

 expected_uplift

 expected_loss

    m_check


1

        0.92082114

... (19 outputs left out)

        0.09154818

    7.868594e-05

  2.537899e-03

 0.00000000

In [5]:
for i, row in py.iterrows():
    r = r_out.iloc[i]
    checks.add(f"P(B>A)      {row['name']}", row.prob_beat_control, r.prob_beat_control, atol=3e-3)
    checks.add(f"E[(B-A)+]   {row['name']}", row.expected_uplift, r.expected_uplift, atol=1e-6, rtol=0.02)
    checks.add(f"E[(A-B)+]   {row['name']}", row.expected_loss, r.expected_loss, atol=1e-6, rtol=0.02)
    if not np.isnan(row.m):
        checks.add(f"lift prior mean {row['name']}", row.m, r.m_check, rtol=1e-12)

## 2. Probability of being best, 3 arms

**No lift prior** (`lift_prior=None`, the default): the importance weights are uniform, so the
exact unweighted integral is the reference.

**Skeptical lift prior**: each challenger's log-lift vs control gets its own normal density and
the joint prior is their product. Given the control rate `a`, the two challengers are then
independent, so

`P(B best) = ∫ f_A(a) ∫_{b > a} g_B(b | a) · G_C(b | a) db da / Z`

where `g_X(t | a) = dbeta_X(t) · dnorm(log t − log a; m, s)`, `G_X(t | a) = ∫_0^t g_X`, and
`Z = ∫ f_A(a) · G_B(1 | a) · G_C(1 | a) da`.

In [6]:
n3, x3 = [15_000, 15_000, 15_000], [450, 495, 480]
res3 = BayesianEngine(seed=2).run_probability_analysis(
    ExperimentInput(visitors=n3, conversions=x3), n_samples=N_SAMPLES)
best_py = [r.prob_being_best for r in res3]  # challengers 1 and 2

prior3 = get_lift_prior(0.0, "skeptical")
m3, s3 = prior3.mean_log_lift, prior3.std_log_lift
res3_skep = BayesianEngine(seed=4).run_probability_analysis(
    ExperimentInput(visitors=n3, conversions=x3), lift_prior=prior3, n_samples=N_SAMPLES)
best_skep_py = [r.prob_being_best for r in res3_skep]
{"no lift prior": best_py, "skeptical": best_skep_py,
 "skeptical ESS": res3_skep[0].effective_sample_size}

{'no lift prior': [0.6674019999983541, 0.2989589999989194],
 'skeptical': [0.6386385161358619, 0.30448435543914176],
 'skeptical ESS': 728241.4192303943}

In [7]:
%%R -i n3 -i x3 -o best_r
n <- unlist(n3); x <- unlist(x3)
a <- 1 + x; b <- 1 + n - x
p_best <- sapply(seq_along(n), function(i) {
  integrate(function(t) {
    out <- dbeta(t, a[i], b[i])
    for (j in setdiff(seq_along(n), i)) out <- out * pbeta(t, a[j], b[j])
    out
  }, qbeta(1e-10, a[i], b[i]), qbeta(1 - 1e-10, a[i], b[i]), rel.tol = 1e-10)$value
})
best_r <- p_best
best_r

[1]

 0.03349094

 0.66665131

 0.29985775

In [8]:
%%R -i n3 -i x3 -i m3 -i s3 -o best_skep_r
n <- unlist(n3); x <- unlist(x3)
pr  <- cbind(1 + x, 1 + n - x)
rng <- t(apply(pr, 1, function(p) qbeta(c(1e-10, 1 - 1e-10), p[1], p[2])))
# Weighted challenger density g_k(t | a) and its integral G_k(t | a) from 0 to t.
g <- function(k, t, a) dbeta(t, pr[k, 1], pr[k, 2]) * dnorm(log(t) - log(a), m3, s3)
G <- function(k, t, a) {
  u <- min(t, rng[k, 2])
  if (u <= rng[k, 1]) return(0)
  integrate(function(v) g(k, v, a), rng[k, 1], u, rel.tol = 1e-8)$value
}
# P(challenger k beats control and the other challenger o | a), unnormalised.
best_given_a <- function(a, k, o) {
  l <- max(a, rng[k, 1]); u <- rng[k, 2]
  if (l >= u) return(0)
  integrate(function(b) g(k, b, a) * sapply(b, function(t) G(o, t, a)),
            l, u, rel.tol = 1e-7)$value
}
over_a <- function(fun) {
  integrate(function(a) dbeta(a, pr[1, 1], pr[1, 2]) * sapply(a, fun),
            rng[1, 1], rng[1, 2], rel.tol = 1e-6)$value
}
z <- over_a(function(a) G(2, Inf, a) * G(3, Inf, a))
best_skep_r <- c(over_a(function(a) best_given_a(a, 2, 3)),
                 over_a(function(a) best_given_a(a, 3, 2))) / z
best_skep_r

[1]

 0.6384751

 0.3046042

In [9]:
for i in (1, 2):
    checks.add(f"P(best) arm {i} of 3, no lift prior", best_py[i - 1], best_r[i], atol=3e-3)
    checks.add(f"P(best) arm {i} of 3, skeptical", best_skep_py[i - 1], best_skep_r[i - 1], atol=3e-3)

## 3. Monetary projection (`run_monetary_projection`)

The engine compares conversion **rates** at equal traffic: each arm gets
`mean(n_A, n_B) / runtime_days` visitors per day, and each arm is valued at its own AOV. The
per-draw AOV noise is one log-normal factor with mean 1, shared by both arms and independent of
the conversion rates, so it changes neither the sign of the revenue difference nor its expected
positive part. Therefore

`expected_uplift = projection_period · E_w[((b·AOV_B − a·AOV_A) · mean(n_A, n_B) / runtime_days)⁺]`

and the same with the negative part for risk. `prob_beat_control` stays rate-based
(P(b > a)); `prob_revenue_better` is P(b·AOV_B > a·AOV_A). R computes all of these exactly.

The unequal-traffic cases (10,000 vs 11,000 visitors, 300 conversions each: equal volumes, lower
rate for B) are where the rate formula differs from the old volume formula
`AOV_B · E_w[(b·n_B − a·n_A)⁺] / runtime_days`. The equal-rate case with a 20% higher AOV for B
is where per-arm AOVs matter: the old formula valued only the rate difference and showed no
expected gain. The `old_volume_uplift` column shows the old volume formula's value for
comparison only; it is not checked.

In [10]:
runtime, period = 14, 183
mon_cases = [
    # name, control n, control x, challenger n, challenger x, (AOV_A, AOV_B), lift prior
    ("near-equal traffic, moderate",    9_000, 270,  9_100, 300, (64.0, 66.5), get_lift_prior(0.0, "moderate")),
    ("unequal traffic, moderate",      10_000, 300, 11_000, 300, (64.0, 66.5), get_lift_prior(0.0, "moderate")),
    ("unequal traffic, no lift prior", 10_000, 300, 11_000, 300, (64.0, 66.5), None),
    ("equal rates, AOV +20%",          10_000, 300, 10_000, 300, (50.0, 60.0), get_lift_prior(0.0, "moderate")),
]
rows = []
for name, na, xa, nb, xb, (aov_a, aov_b), prior in mon_cases:
    mon = BayesianEngine(seed=3).run_monetary_projection(
        visitors=[na, nb], conversions=[xa, xb],
        biz_case=BusinessCaseInput(aovs={"Control": aov_a, "Variant": aov_b},
                                   runtime_days=runtime, projection_period=period),
        prob_best_overall=[0.0, 0.0], variant_labels=["Control", "Variant"],
        lift_prior=prior, n_simulations=N_SAMPLES,
    )[0]
    rows.append(dict(name=name, na=na, xa=xa, nb=nb, xb=xb, aov_a=aov_a, aov_b=aov_b,
                     m=prior.mean_log_lift if prior else np.nan,
                     s=prior.std_log_lift if prior else np.nan,
                     **{k: mon[k] for k in ["prob_beat_control", "prob_revenue_better",
                                            "expected_uplift", "expected_risk"]}))
mon_py = pd.DataFrame(rows)
mon_py

,name,na,xa,nb,xb,aov_a,aov_b,m,s,prob_beat_control,prob_revenue_better,expected_uplift,expected_risk
0,"near-equal traffic, moderate",9000,270,9100,300,64.0,66.5,0.0,0.25,0.861285,0.942221,30482.16,465.90
1,"unequal traffic, moderate",10000,300,11000,300,64.0,66.5,0.0,0.25,0.129333,0.265601,3168.94,15469.14
2,"unequal traffic, no lift prior",10000,300,11000,300,64.0,66.5,NaN,NaN,0.117770,0.239785,2909.19,17492.92
3,"equal rates, AOV +20%",10000,300,10000,300,50.0,60.0,0.0,0.25,0.501050,0.991503,39387.60,46.81


In [11]:
%%R -i mon_py -i runtime -i period -o mon_r
mon_r <- do.call(rbind, lapply(seq_len(nrow(mon_py)), function(i) {
  r <- mon_py[i, ]
  pa <- c(1 + r$xa, 1 + r$na - r$xa); pb <- c(1 + r$xb, 1 + r$nb - r$xb)
  z <- nested(pa, pb, r$m, r$s)
  E <- function(...) nested(pa, pb, r$m, r$s, ...) / z
  traffic <- mean(c(r$na, r$nb)) / runtime            # daily visitors per arm
  k <- r$aov_a / r$aov_b                              # b * AOV_B > a * AOV_A  <=>  b > a * k
  rev <- function(a, b) (b * r$aov_b - a * r$aov_a) * traffic
  ratio <- r$na / r$nb                                # old formula: b * nb > a * na
  data.frame(
    prob_beat_control   = E(lo = function(a) a),
    prob_revenue_better = E(lo = function(a) a * k),
    expected_uplift     = period * E(h = rev, lo = function(a) a * k),
    expected_risk       = period * E(h = function(a, b) -rev(a, b), hi = function(a) a * k),
    old_volume_uplift   = period * r$aov_b / runtime *
      E(h = function(a, b) b * r$nb - a * r$na, lo = function(a) a * ratio)
  )
}))
mon_r

 prob_beat_control

 prob_revenue_better

 expected_uplift

 expected_risk


1

         0.8608648

... (23 outputs left out)


3

          8382.694


4

          7193.859

In [12]:
for i, row in mon_py.iterrows():
    r = mon_r.iloc[i]
    checks.add(f"monetary P(B>A)      {row['name']}", row.prob_beat_control, r.prob_beat_control, atol=3e-3)
    checks.add(f"monetary P(rev B>A)  {row['name']}", row.prob_revenue_better, r.prob_revenue_better, atol=3e-3)
    checks.add(f"monetary uplift      {row['name']}", row.expected_uplift, r.expected_uplift, rtol=0.02)
    checks.add(f"monetary risk        {row['name']}", row.expected_risk, r.expected_risk, atol=1.0, rtol=0.03)

## Summary

In [13]:
checks.report()

39 checks, 39 passed, 0 failed

Largest differences:
                                              check  hexkit       R  abs_diff             tolerance
monetary risk        unequal traffic, no lift prior 17492.9 17541.1   48.2042     atol=1, rtol=0.03
     monetary risk        unequal traffic, moderate 15469.1 15510.2   41.0798     atol=1, rtol=0.03
  monetary uplift      near-equal traffic, moderate 30482.2 30457.6   24.5194 atol=1e-08, rtol=0.02
monetary uplift      unequal traffic, no lift prior 2909.19 2901.52   7.66819 atol=1e-08, rtol=0.02
     monetary uplift      unequal traffic, moderate 3168.94  3162.5   6.43828 atol=1e-08, rtol=0.02


,check,hexkit,R,abs_diff,tolerance,pass
0,"P(B>A) small positive, no lift prior",0.920780,0.920821,4.113863e-05,"atol=0.003, rtol=0",True
1,"E[(B-A)+] small positive, no lift prior",0.003587,0.003588,8.905278e-07,"atol=1e-06, rtol=0.02",True
2,"E[(A-B)+] small positive, no lift prior",0.000089,0.000089,3.165154e-07,"atol=1e-06, rtol=0.02",True
3,"P(B>A) small positive, uninformative",0.920146,0.920187,4.119055e-05,"atol=0.003, rtol=0",True
4,"E[(B-A)+] small positive, uninformative",0.003567,0.003568,9.455795e-07,"atol=1e-06, rtol=0.02",True
5,"E[(A-B)+] small positive, uninformative",0.000090,0.000090,3.183132e-07,"atol=1e-06, rtol=0.02",True
6,"lift prior mean small positive, uninformative",0.000000,0.000000,0.000000e+00,"atol=1e-08, rtol=1e-12",True
7,"P(B>A) small positive, skeptical",0.866922,0.866881,4.090731e-05,"atol=0.003, rtol=0",True
8,"E[(B-A)+] small positive, skeptical",0.002305,0.002306,1.531863e-06,"atol=1e-06, rtol=0.02",True
9,"E[(A-B)+] small positive, skeptical",0.000132,0.000131,3.100307e-07,"atol=1e-06, rtol=0.02",True
